# Curriculum experiment — Round 3: base model, where fine-tuning must help

**The story so far:**
- Round 1 (Instruct, LR 2e-4): fine-tuning hurt. baseline 43.0% > A 34.0% > C 32.0% > B 30.5%
- Round 2 (Instruct, LR 2e-5 + prompt masking): still hurt, less. baseline 43.0% > A 36.0% > B 33.5% > C 29.0%
- Emerging pattern: **shuffled beat both orderings, twice.** But we couldn't judge fairly because training was net-harmful — the Instruct model already knew GSM8K-style math better than GSM8K's own terse answers.

**Round 3 changes:**
1. **Base model** (`Qwen/Qwen2.5-0.5B`, not Instruct) — never instruction-tuned, so its zero-shot GSM8K score should be low and fine-tuning should unambiguously help
2. **Plain Question/Answer format** instead of chat template (base models have no dialogue training)
3. **LR 1e-4** — a base model has to actually learn the task; 2e-5 may be too timid. Constant LR kept for ordering fairness. Prompt masking kept from round 2.

Same 2,000 training problems, same three orderings, same 200 test problems as rounds 1–2.

**The question, at last:** in a regime where fine-tuning clearly helps, does shuffled still beat both orderings?

Setup: T4 GPU, Run all, ~1.5–2 hours.

In [1]:
# Cell 1: Install
!pip uninstall -y -q torchao
!pip install -q -U transformers datasets peft accelerate

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00
GPU: Tesla T4


In [2]:
# Cell 2: Data — identical to rounds 1 and 2
import random
from datasets import load_dataset

SEED = 42
N_TRAIN = 2000
N_EVAL = 200

gsm = load_dataset("openai/gsm8k", "main")

def difficulty(example):
    return example["answer"].count("<<")

train_pool = list(gsm["train"])
random.Random(SEED).shuffle(train_pool)
train_pool = train_pool[:N_TRAIN]
eval_pool = list(gsm["test"])[:N_EVAL]

order_A = list(train_pool)
random.Random(SEED + 1).shuffle(order_A)
order_B = sorted(train_pool, key=difficulty)
order_C = sorted(train_pool, key=difficulty, reverse=True)

print(f"Train: {len(train_pool)}, eval: {len(eval_pool)} — same data as rounds 1-2")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train: 2000, eval: 200 — same data as rounds 1-2


In [3]:
# Cell 3: Tokenization — plain Question/Answer format (no chat template), prompt masked
import re
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # BASE model, not Instruct
MAX_LEN = 640

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt(question):
    return f"Question: {question}\nAnswer:"

def tokenize_masked(ex):
    answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
    prompt = make_prompt(ex["question"])
    full = prompt + " " + answer + tokenizer.eos_token  # EOS teaches it to STOP

    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
    enc = tokenizer(full, truncation=True, max_length=MAX_LEN)

    labels = list(enc["input_ids"])
    n_prompt = min(len(prompt_ids), len(labels))
    labels[:n_prompt] = [-100] * n_prompt

    return {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": labels}

dataset_A = [tokenize_masked(e) for e in order_A]
dataset_B = [tokenize_masked(e) for e in order_B]
dataset_C = [tokenize_masked(e) for e in order_C]

ex0 = dataset_A[0]
n_masked = sum(1 for l in ex0["labels"] if l == -100)
print(f"Example 0: {len(ex0['input_ids'])} tokens, {n_masked} masked, {len(ex0['input_ids'])-n_masked} trained")
assert 0 < n_masked < len(ex0["labels"]), "Masking looks wrong!"
print("✓ Prompt masking active")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Example 0: 115 tokens, 54 masked, 61 trained
✓ Prompt masking active


In [4]:
# Cell 4: Training — LR 1e-4 constant, order preserved, masked collator
import gc
import torch.nn as nn
from torch.utils.data import Dataset, SequentialSampler
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

class ListDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

class OrderedTrainer(Trainer):
    def _get_train_sampler(self, *args, **kwargs):
        return SequentialSampler(self.train_dataset)

def collate_masked(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, attn, labels = [], [], []
    for x in batch:
        n = max_len - len(x["input_ids"])
        input_ids.append(list(x["input_ids"]) + [pad_id] * n)
        attn.append(list(x["attention_mask"]) + [0] * n)
        labels.append(list(x["labels"]) + [-100] * n)
    return {"input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels)}

def train_one_run(run_name, dataset):
    print(f"\n{'='*50}\nTraining run {run_name}\n{'='*50}")
    torch.manual_seed(SEED)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model.config.use_cache = False

    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, lora)

    args = TrainingArguments(
        output_dir=f"./out_{run_name}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=1e-4,             # base model must actually learn the task
        lr_scheduler_type="constant",
        warmup_steps=10,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    trainer = OrderedTrainer(model=model, args=args,
                             train_dataset=ListDataset(dataset),
                             data_collator=collate_masked)
    trainer.train()

    model.save_pretrained(f"./adapter_r3_{run_name}")
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    print(f"Run {run_name} done — saved to ./adapter_r3_{run_name}")

In [5]:
# Cell 5: Order-preservation sanity check
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, **kwargs):
        return None

test_trainer = OrderedTrainer(
    model=DummyModel(),
    args=TrainingArguments(output_dir="./tmp", report_to="none"),
    train_dataset=ListDataset(list(range(100))),
)
try:
    sampler = test_trainer._get_train_sampler()
except TypeError:
    sampler = test_trainer._get_train_sampler(test_trainer.train_dataset)
order = list(sampler)
assert order == list(range(100)), f"ORDER BROKEN! First 10: {order[:10]}"
print("✓ Order preserved — experiment is valid")

✓ Order preserved — experiment is valid


In [6]:
# Cell 6: Run all three trainings (resume-friendly)
import os
for name, ds in [("A_shuffled", dataset_A), ("B_easy_to_hard", dataset_B), ("C_hard_to_easy", dataset_C)]:
    if os.path.exists(f"./adapter_r3_{name}"):
        print(f"Skipping {name} — adapter exists")
    else:
        train_one_run(name, ds)


Training run A_shuffled


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

...

Step,Training Loss
50,0.638737
100,0.595676
150,0.541245
200,0.526334
250,0.523211


Run B_easy_to_hard done — saved to ./adapter_r3_B_easy_to_hard

Training run C_hard_to_easy


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
50,0.550204
100,0.544136
150,0.539426
200,0.557478
250,0.610858


Run C_hard_to_easy done — saved to ./adapter_r3_C_hard_to_easy


In [7]:
# Cell 7: Evaluation — Question/Answer prompting, cut runaway generations at 'Question:'
from peft import PeftModel

def extract_gold(answer_text):
    return answer_text.split("####")[-1].strip().replace(",", "")

def extract_pred(generated_text):
    # Base models may keep generating new questions — keep only the first answer
    generated_text = generated_text.split("Question:")[0]
    if "####" in generated_text:
        tail = generated_text.split("####")[-1]
        nums = re.findall(r"-?\d+\.?\d*", tail.replace(",", ""))
        if nums: return nums[0].rstrip(".")
    nums = re.findall(r"-?\d+\.?\d*", generated_text.replace(",", ""))
    return nums[-1].rstrip(".") if nums else None

def norm(x):
    try: return float(x)
    except (TypeError, ValueError): return None

@torch.no_grad()
def evaluate(adapter_path, label, batch_size=8):
    print(f"\nEvaluating {label} ...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    model.eval()

    tokenizer.padding_side = "left"
    correct = 0
    for i in range(0, len(eval_pool), batch_size):
        batch = eval_pool[i:i+batch_size]
        prompts = [make_prompt(ex["question"]) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to("cuda")
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j, ex in enumerate(batch):
            gen = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            if norm(extract_pred(gen)) is not None and norm(extract_pred(gen)) == norm(extract_gold(ex["answer"])):
                correct += 1
        if (i // batch_size) % 5 == 0:
            print(f"  {i+len(batch)}/{len(eval_pool)} — running accuracy: {correct/(i+len(batch)):.1%}")

    del model, base
    gc.collect(); torch.cuda.empty_cache()
    acc = correct / len(eval_pool)
    print(f"{label}: {correct}/{len(eval_pool)} = {acc:.1%}")
    return acc

results = {}
results["baseline (no fine-tune)"] = evaluate(None, "baseline (no fine-tune)")
results["A: shuffled"]             = evaluate("./adapter_r3_A_shuffled", "A: shuffled")
results["B: easy \u2192 hard"]       = evaluate("./adapter_r3_B_easy_to_hard", "B: easy → hard")
results["C: hard \u2192 easy"]       = evaluate("./adapter_r3_C_hard_to_easy", "C: hard → easy")


Evaluating baseline (no fine-tune) ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 12.5%
  48/200 — running accuracy: 6.2%
  88/200 — running accuracy: 6.8%
  128/200 — running accuracy: 5.5%
  168/200 — running accuracy: 6.0%
baseline (no fine-tune): 13/200 = 6.5%

Evaluating A: shuffled ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 37.5%
  48/200 — running accuracy: 31.2%
  88/200 — running accuracy: 34.1%
  128/200 — running accuracy: 34.4%
  168/200 — running accuracy: 33.9%
A: shuffled: 66/200 = 33.0%

Evaluating B: easy → hard ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 25.0%
  48/200 — running accuracy: 29.2%
  88/200 — running accuracy: 31.8%
  128/200 — running accuracy: 33.6%
  168/200 — running accuracy: 33.9%
B: easy → hard: 67/200 = 33.5%

Evaluating C: hard → easy ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 25.0%
  48/200 — running accuracy: 37.5%
  88/200 — running accuracy: 36.4%
  128/200 — running accuracy: 33.6%
  168/200 — running accuracy: 33.9%
C: hard → easy: 64/200 = 32.0%


In [8]:
# Cell 8: All three rounds side by side
round1 = {"baseline (no fine-tune)": 0.430, "A: shuffled": 0.340,
          "B: easy \u2192 hard": 0.305, "C: hard \u2192 easy": 0.320}
round2 = {"baseline (no fine-tune)": 0.430, "A: shuffled": 0.360,
          "B: easy \u2192 hard": 0.335, "C: hard \u2192 easy": 0.290}

print("\n" + "="*70)
print("RESULTS — GSM8K accuracy, 200 held-out problems")
print("="*70)
print(f"{'':<26}{'R1 Instruct':>13}{'R2 Instruct':>13}{'R3 BASE':>13}")
print(f"{'':<26}{'LR 2e-4':>13}{'2e-5 masked':>13}{'1e-4 masked':>13}")
print("-"*70)
for k in round1:
    r3 = results.get(k)
    print(f"{k:<26}{round1[k]:>12.1%}{round2[k]:>13.1%}{r3:>13.1%}")
print("="*70)
print("\nInterpretation:")
print("1. Baseline should be LOW here (base model, no instruction tuning).")
print("2. A > baseline?  Should finally be YES — training a fresh model on a")
print("   task it doesn't know must help. If not, something is broken; stop.")
print("3. THE question: does A (shuffled) beat B and C again? Three-for-three")
print("   would be a tidy finding: ordering by difficulty hurts small-model SFT.")
print("   If B or C wins here instead, ordering effects depend on regime —")
print("   also interesting, and worth a second seed to confirm.")


RESULTS — GSM8K accuracy, 200 held-out problems
                            R1 Instruct  R2 Instruct      R3 BASE
                                LR 2e-4  2e-5 masked  1e-4 masked
----------------------------------------------------------------------
baseline (no fine-tune)          43.0%        43.0%         6.5%
A: shuffled                      34.0%        36.0%        33.0%
B: easy → hard                   30.5%        33.5%        33.5%
C: hard → easy                   32.0%        29.0%        32.0%

Interpretation:
1. Baseline should be LOW here (base model, no instruction tuning).
2. A > baseline?  Should finally be YES — training a fresh model on a
   task it doesn't know must help. If not, something is broken; stop.
3. THE question: does A (shuffled) beat B and C again? Three-for-three
   would be a tidy finding: ordering by difficulty hurts small-model SFT.
   If B or C wins here instead, ordering effects depend on regime —
   also interesting, and worth a second seed to co

In [ ]:
from google.colab import drive
drive.mount('/content/drive')